### CEO App

Deploys the CEO Dashboard Databricks App. Grants SQL warehouse access, wires the
Lakebase database resource, and deploys the FastAPI app from apps/ceo-dashboard/.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
CEO_PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-ceo-sessions".lower())
CEO_ENDPOINT_PATH = f"projects/{CEO_PROJECT_ID}/branches/production/endpoints/ep-primary"

In [ ]:
import sys, os
sys.path.append('../utils')
from uc_state import add

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import (
    App, AppResource, AppResourceSqlWarehouse,
    AppResourceSqlWarehouseSqlWarehousePermission,
    AppDeployment,
)
from databricks.sdk.service import catalog as catalog_svc

w = WorkspaceClient()

# Reuse or create warehouse
WAREHOUSE_NAME = f"{CATALOG}-ceo-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"\u267b\ufe0f Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"\u2705 Created warehouse: {warehouse.id}")

In [ ]:
source_code_path = os.path.abspath("../apps/ceo-dashboard")
APP_NAME = f"ceo-dashboard"

# Lakebase Autoscale does not support AppResourceDatabase — the app connects
# directly using w.postgres.generate_database_credential() with the endpoint path.
app_def = App(
    name=APP_NAME,
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE,
            ),
        ),
    ],
)

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"\u267b\ufe0f App {APP_NAME} exists, updating...")
    app_status = w.apps.update(APP_NAME, app_def)  # returns App directly
except Exception:
    app_status = w.apps.create(app_def).result()  # returns Wait[App]

add(CATALOG, "apps", app_status)
print(f"\u2705 App {APP_NAME} ready")

# Resolve SP ID immediately — used in all subsequent permission cells.
# service_principal_client_id is the documented OAuth UUID field.
app_sp_id = (
    getattr(app_status, 'service_principal_client_id', None)
    or (app_status.as_dict() if hasattr(app_status, 'as_dict') else {}).get('service_principal_client_id')
    or (app_status.as_dict() if hasattr(app_status, 'as_dict') else {}).get('id')
)
assert app_sp_id, "Could not determine app service principal ID"
print(f"\U0001f511 App SP ID: {app_sp_id}")

In [ ]:
# Grant app permissions to relevant UC resources
for full_name, securable_type, privilege in [
    (CATALOG, "CATALOG", catalog_svc.Privilege.USE_CATALOG),
    (f"{CATALOG}.lakeflow", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    # all_events raw table
    (f"{CATALOG}.lakeflow.all_events", "TABLE", catalog_svc.Privilege.SELECT),
    # Gold/silver tables queried by Genie revenue & ops spaces
    (f"{CATALOG}.lakeflow.gold_brand_sales_day", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.gold_location_sales_hourly", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.gold_order_header", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.lakeflow.silver_order_items", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    # Simulator dimension tables queried by Genie spaces
    (f"{CATALOG}.simulator.brand_locations", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.brands", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.items", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.simulator.locations", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.food_safety", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.food_safety.inspections", "TABLE", catalog_svc.Privilege.SELECT),
    (f"{CATALOG}.food_safety.violations", "TABLE", catalog_svc.Privilege.SELECT),
    # Document schemas — needed so the app can list PDFs from UC volumes
    (f"{CATALOG}.legal_complaints", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.regulatory", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.audits", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.consultancy", "SCHEMA", catalog_svc.Privilege.USE_SCHEMA),
    (f"{CATALOG}.legal_complaints", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.regulatory", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.audits", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
    (f"{CATALOG}.consultancy", "SCHEMA", catalog_svc.Privilege.READ_VOLUME),
]:
    try:
        w.grants.update(
            full_name=full_name,
            securable_type=securable_type,
            changes=[
                catalog_svc.PermissionsChange(
                    add=[privilege],
                    principal=app_sp_id,
                )
            ],
        )
        print(f"\u2705 Granted {privilege} on {full_name}")
    except Exception as e:
        print(f"\u26a0\ufe0f Could not grant {privilege} on {full_name}: {e}")

In [ ]:
import json, time
from databricks.sdk.service.serving import (
    ServingEndpointAccessControlRequest,
    ServingEndpointPermissionLevel,
)

def _grant_can_query(endpoint_id: str, endpoint_name: str, sp_id: str, retries: int = 3) -> bool:
    """Grant CAN_QUERY on a serving endpoint.
    endpoint_id  — the UUID returned by the list API (required by update_permissions)
    endpoint_name — human-readable name used only for logging
    """
    for attempt in range(retries):
        try:
            w.serving_endpoints.update_permissions(
                serving_endpoint_id=endpoint_id,   # must be UUID, not name
                access_control_list=[
                    ServingEndpointAccessControlRequest(
                        service_principal_name=sp_id,
                        permission_level=ServingEndpointPermissionLevel.CAN_QUERY,
                    )
                ],
            )
            print(f"\u2705 Granted CAN_QUERY on {endpoint_name} ({endpoint_id})")
            return True
        except Exception as e:
            if attempt < retries - 1:
                print(f"  Retry {attempt+1}/{retries} for {endpoint_name}: {e}")
                time.sleep(5 * (attempt + 1))
            else:
                print(f"\u274c FAILED to grant CAN_QUERY on {endpoint_name}: {e}")
                return False

def _grant_can_run_genie(space_id: str, sp_id: str, retries: int = 3) -> bool:
    """Grant CAN_RUN on a Genie space. Returns True on success."""
    for attempt in range(retries):
        try:
            w.api_client.do(
                "PATCH",
                f"/api/2.0/permissions/genie/{space_id}",
                body={"access_control_list": [
                    {"service_principal_name": sp_id, "permission_level": "CAN_RUN"}
                ]},
            )
            print(f"\u2705 Granted CAN_RUN on Genie space {space_id} to {sp_id}")
            return True
        except Exception as e:
            if attempt < retries - 1:
                print(f"  Retry {attempt+1}/{retries} for Genie space {space_id}: {e}")
                time.sleep(5 * (attempt + 1))
            else:
                print(f"\u274c FAILED to grant CAN_RUN on Genie space {space_id}: {e}")
                return False

# ── Grant CAN_QUERY on ALL mas-*/ka-* serving endpoints ───────────────────────
# Scan by name pattern — reliable regardless of uc_state contents.
# IMPORTANT: update_permissions requires the endpoint UUID (ep.id), not the name.
ep_failures = []
all_eps = list(w.serving_endpoints.list())
target_eps = [ep for ep in all_eps if (ep.name or "").startswith(("mas-", "ka-"))]
print(f"Found {len(target_eps)} mas/ka endpoints to grant")
for ep in target_eps:
    ok = _grant_can_query(ep.id, ep.name, app_sp_id)
    if not ok:
        ep_failures.append(ep.name)

# ── Grant CAN_RUN on ALL Genie spaces ─────────────────────────────────────────
genie_failures = []
all_spaces = w.api_client.do("GET", "/api/2.0/genie/spaces").get("spaces", [])
print(f"Found {len(all_spaces)} Genie spaces to grant")
for space in all_spaces:
    space_id = space.get("space_id", "")
    if space_id:
        ok = _grant_can_run_genie(space_id, app_sp_id)
        if not ok:
            genie_failures.append(space_id)

# ── Fail loudly if any grants didn't land ─────────────────────────────────────
if ep_failures or genie_failures:
    raise RuntimeError(
        f"Permission grants failed!\n"
        f"  Endpoints: {ep_failures}\n"
        f"  Genie spaces: {genie_failures}\n"
        "Fix the errors above and re-run this cell."
    )
print("\n\u2705 All endpoint and Genie space permissions granted successfully")

In [ ]:
# Grant DATABRICKS_SUPERUSER to the app service principal in the Lakebase Autoscale project.
# This allows the app to connect and create tables on first startup.
try:
    from databricks.sdk.service.postgres import Role, RoleRoleSpec, RoleMembershipRole, RoleIdentityType
    from databricks.sdk.common.types.fieldmask import FieldMask

    production_branch = f"projects/{CEO_PROJECT_ID}/branches/production"
    app_principal = app_sp_id  # resolved service principal client ID

    # Find or create a role for the app SP
    existing_roles = list(w.postgres.list_roles(production_branch))
    app_role = next(
        (r for r in existing_roles if getattr(r.spec, "postgres_role", None) == app_principal),
        None,
    )

    if app_role:
        app_role.spec = RoleRoleSpec(
            identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
            postgres_role=app_principal,
            membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
        )
        w.postgres.update_role(
            name=app_role.name,
            role=app_role,
            update_mask=FieldMask(field_mask=["spec.membership_roles"]),
        )
        print(f"\u2705 Updated app role to DATABRICKS_SUPERUSER")
    else:
        new_role = w.postgres.create_role(
            parent=production_branch,
            role=Role(
                spec=RoleRoleSpec(
                    identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                    postgres_role=app_principal,
                    membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
                ),
            ),
        )
        print(f"\u2705 Created DATABRICKS_SUPERUSER role for app SP: {app_principal}")
except Exception as e:
    print(f"\u26a0\ufe0f Could not grant Lakebase role to app SP: {e}")

##### Write app.yaml — assembles all resource IDs from uc_state before deploy

In [ ]:
import json as _json, re as _re, os as _os

def _latest_from_uc_state(resource_type, key):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = '{resource_type}'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            val = _json.loads(row.resource_data).get(key, "")
            if val:
                return val
    except Exception as e:
        print(f"⚠️ uc_state lookup failed for {resource_type}/{key}: {e}")
    return ""

def _ka_tile_id(ka_name):
    try:
        df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = 'knowledge_assistants'
            ORDER BY created_at DESC
        """)
        for row in df.collect():
            info = _json.loads(row.resource_data)
            if info.get("name") == ka_name:
                return info.get("tile_id", "")
    except Exception as e:
        print(f"⚠️ Could not read KA tile {ka_name}: {e}")
    return ""

# Supervisor
supervisor_endpoint = _latest_from_uc_state("multi_agent_supervisors", "endpoint_name")
supervisor_tile_id  = _latest_from_uc_state("multi_agent_supervisors", "tile_id")

supervisor_mlflow_experiment_id = ""
if supervisor_tile_id:
    try:
        _tile = w.api_client.do("GET", f"/api/2.0/tiles/{supervisor_tile_id}")
        supervisor_mlflow_experiment_id = str(_tile.get("mlflow_experiment_id") or "")
    except Exception as e:
        print(f"⚠️ Could not fetch mlflow_experiment_id from tile: {e}")

# Genie spaces
_revenue_title = f"CEO Revenue & Orders Intelligence ({CATALOG})"
_ops_title     = f"CEO Operations Intelligence ({CATALOG})"
genie_revenue_id = genie_ops_id = ""
try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'genie_spaces'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = _json.loads(row.resource_data)
        if info.get("title") == _revenue_title and not genie_revenue_id:
            genie_revenue_id = info.get("space_id", "")
        if info.get("title") == _ops_title and not genie_ops_id:
            genie_ops_id = info.get("space_id", "")
except Exception as e:
    print(f"⚠️ Could not read genie IDs: {e}")

# KA tile IDs
ka_inspection  = _ka_tile_id(f"{CATALOG}-inspection-knowledge")
ka_legal       = _ka_tile_id(f"{CATALOG}-ceo-legal")
ka_regulatory  = _ka_tile_id(f"{CATALOG}-ceo-regulatory")
ka_audits      = _ka_tile_id(f"{CATALOG}-ceo-audits")
ka_consultancy = _ka_tile_id(f"{CATALOG}-ceo-consultancy")

# Lakebase endpoint path (deterministic from CATALOG)
_project_id   = _re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-ceo-sessions".lower())
lakebase_endpoint_path = f"projects/{_project_id}/branches/production/endpoints/primary"

app_yaml_path = _os.path.abspath("../apps/ceo-dashboard/app.yaml")
app_yaml_contents = f"""command:
  - uvicorn
  - app.main:app
env:
  - name: LAKEBASE_ENDPOINT_PATH
    value: '{lakebase_endpoint_path}'
  - name: LAKEBASE_DATABASE_NAME
    value: 'databricks_postgres'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: CEO_SUPERVISOR_ENDPOINT
    value: '{supervisor_endpoint}'
  - name: CEO_SUPERVISOR_TILE_ID
    value: '{supervisor_tile_id}'
  - name: CEO_SUPERVISOR_MLFLOW_EXPERIMENT_ID
    value: '{supervisor_mlflow_experiment_id}'
  - name: GENIE_ID_REVENUE
    value: '{genie_revenue_id}'
  - name: GENIE_ID_OPS
    value: '{genie_ops_id}'
  - name: KA_ID_INSPECTION
    value: '{ka_inspection}'
  - name: KA_ID_LEGAL
    value: '{ka_legal}'
  - name: KA_ID_REGULATORY
    value: '{ka_regulatory}'
  - name: KA_ID_AUDITS
    value: '{ka_audits}'
  - name: KA_ID_CONSULTANCY
    value: '{ka_consultancy}'
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse.id}'
"""
with open(app_yaml_path, "w") as _f:
    _f.write(app_yaml_contents)

print(f"✅ Wrote app.yaml to {app_yaml_path}")
print(f"   Supervisor endpoint:  {supervisor_endpoint}")
print(f"   Supervisor tile:      {supervisor_tile_id}")
print(f"   MLflow experiment:    {supervisor_mlflow_experiment_id}")
print(f"   Lakebase endpoint:    {lakebase_endpoint_path}")
print(f"   Revenue Genie:        {genie_revenue_id}")
print(f"   Ops Genie:            {genie_ops_id}")
print(f"   KA Inspection:        {ka_inspection}")
print(f"   KA Legal:             {ka_legal}")
print(f"   KA Regulatory:        {ka_regulatory}")
print(f"   KA Audits:            {ka_audits}")
print(f"   KA Consultancy:       {ka_consultancy}")
print(f"   Warehouse:            {warehouse.id}")

In [ ]:
# Deploy the app
deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(source_code_path=source_code_path),
)
deployment_status = deployment.result()
print(f"\u2705 CEO Dashboard deployed")
print(f"   URL: {app_status.url if hasattr(app_status, 'url') else 'Check Databricks Apps UI'}")
display(deployment_status)

In [ ]:
# Seed EMEA locations (5-8) into simulator and lakeflow tables.
# This is idempotent — safe to re-run on every deploy.
# If the lakeflow pipeline hasn't processed events yet the notebook will
# wait up to 5 min for data; if still empty it exits gracefully and you
# can re-run it manually once the pipeline has data.
result = dbutils.notebook.run(
    "./ceo_add_locations",
    timeout_seconds=600,
    arguments={"CATALOG": CATALOG},
)
print(f"\u2705 ceo_add_locations result: {result}")